# Session 7 - Ensembles, Interpretation & Error Analysis

**Block 2: Machine Learning** · 4 hours

---

## Learning objectives

By the end of this session you will be able to:

1. Explain the mechanism by which bagging and boosting reduce error, and predict
   which will overfit under which conditions. `[CLO6]`
2. Tune a boosting model with early stopping under a correct protocol. `[CLO4, CLO6]`
3. Produce and interpret a permutation-importance ranking, stating what it does
   *not* prove. `[CLO10]`
4. Identify a subgroup on which the model underperforms, and propose a response. `[CLO10]`

## Prerequisites

Sessions 1–6. Especially Session 6: today we compare five models, and without the
paired test you would draw the wrong conclusion from four of the comparisons.

## Why does this matter?

Ensembles are where most practical tabular machine learning ends up. But "boosting
usually wins" is a folk belief, and today you will find it is true on one of our two
problems and false on the other.

You will also do the thing that separates a model from a *system*: look at who it
fails for. We are building a tool that directs municipal inspectors to people's
homes. Aggregate accuracy is not sufficient grounds for deploying that.

## §1 - Retrieval practice

From memory. Five minutes.

1. Thirty single splits of one model gave R² from 0.726 to 0.843. What does that mean
   for any single reported score?
2. Why did random KFold report both a higher mean *and* a smaller standard deviation
   than GroupKFold?
3. Why did we remove `minimum_nights` from the price model even though it passes the
   deployment test?
4. Ridge 0.606 vs HistGB 0.591 on the same folds. What test, and what conclusion?
5. What is nested cross-validation for?

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# xgboost is a third library alongside sklearn's two boosters, included so the
# comparison covers the implementation a working team is most likely to reach for.
import xgboost as xgb
from sklearn.compose import ColumnTransformer
# Two families in one import: the RandomForest pair are bagging, the HistGB pair
# are boosting. The difference between those two words is most of this session.
from sklearn.ensemble import (HistGradientBoostingClassifier,
                              HistGradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.impute import SimpleImputer
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import recall_score, roc_auc_score
from sklearn.model_selection import (GroupKFold, GroupShuffleSplit, cross_val_predict,
                                     cross_val_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from src.data import SEED, load_raw, set_seed, split_by_host

sns.set_theme(style="whitegrid")
set_seed(SEED)

df = load_raw()
train, _ = split_by_host(df)
# The same licence rule as Session 5, inlined as a lambda. Same definition, same
# target, so today's numbers are comparable with that session's.
train["licensed"] = train["license"].map(
    lambda s: bool(re.search(r"HUT|HB-|AJ0|ESFC", str(s))) if str(s) != "nan" else False)
train["price_num"] = (
    train["price"].astype(str).str.replace(r"[^0-9.]", "", regex=True)
    .replace("", np.nan).astype(float))

NUMERIC = ["accommodates", "bedrooms", "beds", "bathrooms", "latitude", "longitude",
           "minimum_nights", "number_of_reviews", "review_scores_rating",
           "calculated_host_listings_count", "availability_365"]
CATEGORICAL = ["room_type", "property_type", "neighbourhood_cleansed"]


def make_preprocessor(numeric=NUMERIC, categorical=CATEGORICAL):
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), numeric),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore",
                                                   min_frequency=20,
                                                   sparse_output=False))]), categorical),
    ])


X = train[NUMERIC + CATEGORICAL]
y = train["licensed"].astype(int)
groups = train["host_id"]

# One fold assignment, reused by every model in this notebook. This is what makes
# the paired comparisons in §4 legitimate.
# list() forces the splitter to produce the index arrays once. Every model below
# is then scored on exactly the same five partitions, which is the precondition for
# pairing fold k against fold k in a t-test.
FOLDS = list(GroupKFold(n_splits=5).split(X, y, groups=groups))
print(f"rows {len(train):,}   positive rate {y.mean():.4f}   folds fixed")

## §2 - Bagging, built by hand

An unbounded decision tree overfits badly - we measured AUC 0.780 for one in
Session 5, against 0.908 for a pruned one.

The bagging idea: an overfitted tree is not *biased*, it is **high-variance**. It
gets the answer roughly right on average and wildly wrong on any particular
resample. So train many of them on different resamples of the data and average.

The averaging cancels the noise. It cannot cancel bias, which is why bagging helps
high-variance models and does nothing for a linear model.

### Predict before you run

We will train 30 unbounded trees, each on its own bootstrap resample (sample n rows
**with replacement**), and average their predicted probabilities.

Predict: (a) the average AUC of the individual trees, (b) the AUC of their average.

### Why an ensemble beats its members

![global picture](../assets/diagrams/s07_ensembles/global_picture.png)

![Bagging](../assets/diagrams/s07_ensembles/Bagging.png)

Six people each with a hand on a different part of an elephant, each labelled a model; the elephant labelled *some unknown distribution*. Then the mechanism: one dataset resampled many times, a learner fitted on each resample, the predictions combined.

The first picture is the argument for the whole block. No individual is wrong - each is **partial, and partial in a different direction**. That last clause is the condition under which averaging helps at all: if every model were wrong in the *same* direction, averaging would preserve the error exactly. In a moment we will measure that disagreement directly, as the standard deviation across 30 trees for each listing.

In [ ]:
# TODO: Predictions first.
#
#   (a) mean AUC of the 30 individual trees :
#   (b) AUC of the averaged prediction      :
#
# Then implement bagging by hand:
#   1. Split off a held-out quarter with GroupShuffleSplit.
#   2. For b in range(30): draw a bootstrap sample of the training rows, fit an
#      unbounded DecisionTreeClassifier, predict_proba on the held-out set.
#   3. Report each tree's AUC, and the AUC of the mean prediction.
#   4. Also report how much the trees disagree per listing (sd across trees).

In [ ]:
# The same result as a curve. The shape is the lesson: steep gains at first, then a
# plateau. More trees never hurt a bagged model, they simply stop helping.
fig, ax = plt.subplots(figsize=(9, 3.6))
ks = np.arange(1, 31)
curve = [roc_auc_score(y_te, tree_preds[:k].mean(axis=0)) for k in ks]
ax.plot(ks, curve, "o-", color="steelblue", label="average of k trees")
# The reference line: what one tree achieves on average. Everything above it was
# bought purely by averaging.
ax.axhline(np.mean(tree_aucs), color="indianred", ls="--",
           label=f"mean individual tree ({np.mean(tree_aucs):.3f})")
ax.set(title="Variance reduction by averaging", xlabel="number of trees averaged",
       ylabel="AUC")
ax.legend()
plt.tight_layout()
plt.show()

### What the numbers say

Each tree on its own is **poor: AUC 0.769**, and they disagree with each other
enormously - the standard deviation of the predicted probability for a single
listing, across trees, is **0.228**. On a 0–1 scale that is close to noise.

Averaged, those same 30 trees reach **0.897**. A gain of **+0.13 AUC** from an
operation that added no information whatsoever: same features, same rows, just
resampled and averaged.

The curve also shows where the value is. Going from 1 to 3 trees buys 0.057; from
10 to 30 buys 0.012. **Variance falls roughly as 1/k**, so returns diminish fast.
That is why nobody tunes `n_estimators` carefully in a Random Forest.

### The gap between hand-bagging and a real Random Forest

Our bagged trees reach 0.897. `RandomForest(30)` reaches **0.908** on identical
data. Both average 30 bootstrapped trees, so where does the extra come from?

**Feature subsampling.** A Random Forest considers only a random subset of features
at each split. That deliberately makes each tree *worse* individually, and makes the
trees **less correlated with each other**. Averaging removes only the part of the
error the trees disagree about - so decorrelating them increases how much the
averaging can remove.

> Bagging averages away variance. Random Forests add a second trick - decorrelation
> - that increases how much variance is available to average away.

This is also why bagging a *linear* model is nearly pointless: two linear models on
bootstrap resamples of 12,000 rows are almost identical, so there is nothing to
average out.

## §3 - Boosting, and the opposite failure mode

Bagging trains many models **in parallel** on different data and averages them.
Boosting trains them **in sequence**, each one correcting its predecessors.

For squared error the algorithm is startlingly simple:

1. Start with a constant prediction $F_0(x) = \bar{y}$.
2. Compute the residuals $r_i = y_i - F_m(x_i)$.
3. Fit a small tree $h_m$ to the **residuals**.
4. Update: $F_{m+1}(x) = F_m(x) + \eta\, h_m(x)$, where $\eta$ is the learning rate.
5. Repeat.

Each tree models what the current ensemble is getting *wrong*.

**The deep idea** ([Friedman, 2001](https://doi.org/10.1214/aos/1013203451)): the
residual is the negative gradient of squared-error loss with respect to the
prediction. So step 3 fits a tree to the negative gradient, and step 4 takes a step
in that direction with step size $\eta$. **Boosting is gradient descent, where the
parameter being optimised is the function itself.** For other losses you fit the
gradient of that loss instead, which is why the same machinery does classification.

You met gradient descent in Session 4 and will meet it again in Session 8. It is the
same idea three times.

> **⏱ Runtime note.** The two sweeps below are the slowest cells in this notebook:
> about **145 s** and **110 s** respectively on a four-thread laptop, out of roughly
> **7 minutes** for the whole notebook. This is the most compute-heavy session of the
> course by a wide margin.
>
> Start them and read on while they work. Every number they produce is also written
> out in the markdown that follows, so if a cell is still running when the class
> moves on, you have lost nothing.

### The consequence: adding trees is not safe

Each new tree in a boosted model *reduces training error by construction* - that is
what it was fitted to do. So a boosted model does not converge to a stable answer as
you add trees; it keeps descending, eventually into noise.

A Random Forest has no such dynamic. Each tree is an independent draw, and averaging
more of them can only stabilise the estimate.

### Predict before you run

We will take `n_estimators` ∈ {10, 50, 200, 800} for both a Random Forest and a
gradient booster (learning rate 0.3, early stopping off).

Sketch both curves. Where does each peak?

In [ ]:
# TODO: Sketch your two curves, then run the comparison.
#
# My prediction: RandomForest peaks at n = ___ ; boosting peaks at n = ___

### Opposite behaviour, same action

| n | RandomForest | Boosting (lr = 0.3) |
|---:|---:|---:|
| 10 | 0.9062 | **0.9204** |
| 50 | 0.9188 | 0.9163 |
| 200 | 0.9216 | 0.9141 |
| 800 † | 0.9223 | 0.9100 |

† measured separately; the 800-tree fits are too slow to run in class.

The forest climbs and flattens - 50 → 200 buys 0.003, and 200 → 800 another 0.001.
The booster **peaks at 10 trees and then declines for the rest of the run**, giving
back a full point of AUC by 800 iterations.

Two practical rules follow, and they are not interchangeable:

- **Random Forest:** `n_estimators` is a compute/accuracy trade-off, not a
  regularisation knob. More is never worse, just slower. Set it as high as you can
  afford.
- **Boosting:** `n_estimators` *is* a regularisation knob, and too many overfits.
  **This is what early stopping is for**, and why boosting has it while forests do
  not.

Notice also that boosting at 10 trees ≈ a forest at 800. Boosting extracts far more
from each tree - and pays for it with a hyperparameter you must actually tune.

### Learning rate and number of trees are one coupled decision

In [ ]:
# learning_rate and n_estimators are not independent knobs. A small rate needs many
# rounds; a large rate needs few. The diagonal structure in the table is that fact.
grid = []
for lr in (0.01, 0.3):          # two rows are enough to show the diagonal
    for n in (20, 60, 150):
        auc = cross_val_score(
            Pipeline([("prep", make_preprocessor()),
                      ("model", HistGradientBoostingClassifier(
                          max_iter=n, learning_rate=lr, early_stopping=False,
                          random_state=SEED))]),
            X, y, cv=FOLDS, scoring="roc_auc").mean()
        grid.append({"learning_rate": lr, "n_estimators": n, "AUC": auc})

# pivot turns the long results into a 2x3 table, rates down the side and rounds
# along the top, which is how the interaction becomes visible.
pivot = pd.DataFrame(grid).pivot(index="learning_rate", columns="n_estimators",
                                 values="AUC")
print(pivot.round(4).to_string())

Read the table as a diagonal. A **small** learning rate takes small steps, so it
needs **many** trees and keeps improving for a long time. A **large** learning rate
takes big steps, arrives quickly, and then overshoots into overfitting.

> `learning_rate` and `n_estimators` are not two hyperparameters. They are one
> decision expressed twice, and tuning either without the other wastes your budget.

The standard recipe: fix the learning rate small (0.05, or 0.01 if you can afford
it), then let **early stopping** choose the number of trees on a validation fold.

## §4 - The benchmark, done properly

Five models, **one fixed set of folds**, paired comparisons. The shared folds are
what make the pairing valid - Session 6's machinery, applied for real.

### The two families, side by side

![Ensemble1](../assets/diagrams/s07_ensembles/Ensemble1.png)

Bagging and boosting drawn next to each other: independent learners fitted in parallel and then averaged, against a chain in which each learner is fitted to what the previous ones got wrong.

The independence on one side is why adding trees to a forest is safe - more independent estimates to average never hurts. The dependence on the other is why adding rounds to a booster is not: past some point the corrections start fitting noise, and `n_estimators` becomes a parameter you can overshoot.

In [ ]:
# The full benchmark. Two baselines from earlier sessions and three ensembles, all
# on the fixed FOLDS so nothing differs except the model.
MODELS = {
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "Tree (depth 5)": DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "RandomForest (200)": RandomForestClassifier(n_estimators=200, random_state=SEED,
                                                 n_jobs=-1),
    "HistGB (default)": HistGradientBoostingClassifier(random_state=SEED),
    # subsample and colsample_bytree add randomness per round, which is XGBoost
    # borrowing an idea from bagging to regularise a boosted model.
    "XGBoost (200)": xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=5,
                                       subsample=0.9, colsample_bytree=0.9,
                                       eval_metric="logloss", random_state=SEED,
                                       n_jobs=-1),
}

# One pass per model. We keep the out-of-fold PROBABILITIES, because §5 and §8 both
# need them and re-fitting would triple the cost of this notebook.
#
# Per-fold AUC computed from out-of-fold predictions is identical to what
# cross_val_score would report: every row's prediction comes from a model that never
# saw that row, so slicing the predictions by fold recovers each fold's score exactly.
oof, scores = {}, {}
for name, model in MODELS.items():
    proba = cross_val_predict(
        Pipeline([("prep", make_preprocessor()), ("model", model)]),
        X, y, cv=FOLDS, method="predict_proba")[:, 1]
    oof[name] = proba
    # test is the index array for one fold, so proba[test] is that fold's
    # out-of-fold predictions and the AUC below is that fold's score.
    s = np.array([roc_auc_score(y.iloc[test], proba[test]) for _, test in FOLDS])
    scores[name] = s
    # Compare the +/- column against the gaps between the means before concluding
    # anything from the ranking.
    print(f"  {name:20s} AUC = {s.mean():.4f} +/- {s.std():.4f}   {np.round(s, 3)}")

In [ ]:
# TODO: The means are ranked. Now find out which differences are real.
#
#   1. Identify the best model by mean AUC.
#   2. For every other model, compute the per-fold difference against it.
#   3. Report the mean difference, the SIGNS of the per-fold differences, and a paired
#      t-test p-value.
#   4. Write one sentence per comparison saying whether the difference is real.

### Two real differences, two illusions

| Comparison | diff | signs | p | verdict |
|---|---:|:---:|---:|---|
| HistGB vs LogisticRegression | +0.0317 | +++++ | **0.0011** | real |
| HistGB vs Tree (depth 5) | +0.0141 | +++++ | **0.0034** | real |
| HistGB vs RandomForest (200) | +0.0001 | ++--+ | 0.9746 | noise |
| HistGB vs XGBoost (200) | +0.0022 | ++--+ | 0.1590 | noise |

**Ensembles genuinely beat single models here.** Both wins have the same sign in all
five folds and p < 0.005. This is a real finding - and note the contrast with
Session 6's regression task, where no ensemble beat Ridge.

**Random Forest and gradient boosting are indistinguishable.** 0.9216 versus 0.9216,
a difference of 0.0001, signs disagreeing across folds, p = 0.97. Anyone who told
you boosting reliably beats bagging is generalising from other datasets.

**XGBoost and scikit-learn's booster are indistinguishable.** p = 0.16.

> That last row is why this course teaches **one** boosting implementation. It is not
> a shortcut to save time - on this problem the libraries are the same algorithm with
> different engineering, and we just measured it. Learning three APIs would have
> bought you nothing except three APIs.

## §5 - Resolving Session 5's cliffhanger

In Session 5 we found that the depth-5 tree had the best AUC and **could not name
200 listings**: it emits only ~104 distinct probabilities, so 337 listings tied at
the k=200 cut. Logistic regression ranked finely but scored worse.

We now have two models tied on AUC. Let us ask the operational question.

In [ ]:
# No refitting: these are the predictions computed in §4.
# The granularity question from Session 5, now asked of the ensembles. A model can
# tie on AUC and still be unusable for handing inspectors a ranked list.
granularity = []
for name in ["Tree (depth 5)", "LogisticRegression", "RandomForest (200)",
             "HistGB (default)"]:
    risk = 1 - oof[name]                   # probability of being UNlicensed
    y_unlicensed = 1 - y
    order = np.argsort(-risk)
    top = order[:200]
    cut = risk[top].min()
    granularity.append({
        "model": name,
        "AUC": scores[name].mean(),
        # A forest of 200 trees can only emit multiples of 1/200, so its score
        # resolution is bounded by its size however good its AUC is.
        "distinct_scores": len(np.unique(np.round(risk, 6))),
        "precision@200": y_unlicensed.iloc[top].sum() / 200,
        # Listings sharing the cut-off score. A large number here means the
        # "top 200" was partly settled by row order, which is indefensible when a
        # host asks why their address was on the list.
        "tied_at_cut": int((np.round(risk, 6) == np.round(cut, 6)).sum()),
    })

print(pd.DataFrame(granularity).round(4).to_string(index=False))

### The tie-breaker is not accuracy

| model | AUC | distinct scores | precision@200 | tied at cut |
|---|---:|---:|---:|---:|
| Tree (depth 5) | 0.9075 | 104 | 0.985 | **337** |
| LogisticRegression | 0.8899 | 12,239 | 0.855 | 1 |
| RandomForest (200) | 0.9216 | 444 | 0.990 | **182** |
| HistGB (default) | 0.9216 | 11,336 | 0.985 | **1** |

Random Forest and HistGB are statistically tied on AUC (p = 0.97). But the forest
emits **444** distinct scores against the booster's **11,336**, leaving 182 listings
tied at the inspection cut-off.

Why? A forest averages 200 trees, each contributing a vote from a finite set of
leaves, so the possible averages are a coarse grid. A booster sums hundreds of small
real-valued increments, so its scores are effectively continuous.

**So the choice between two indistinguishable models is decided by an operational
requirement no metric on the sheet measures.** For an enforcement programme where a
host may appeal, the model must be able to say why listing 200 was selected and 201
was not. Only the booster can.

> When two models are statistically tied, stop looking at the metric and look at the
> decision. Something in the deployment will separate them.

**Model Card v1 champion: `HistGradientBoostingClassifier`** - not because it scored
best (it tied), but because it is the only one of the two that produces a defensible
ranking.

## §6 - The industry landscape, in thirty minutes

You have now used two gradient-boosting implementations. There are three you will
hear about constantly.

| | XGBoost (2016) | LightGBM (2017) | CatBoost (2018) |
|---|---|---|---|
| Key idea | regularised objective, second-order optimisation | histogram binning + leaf-wise growth | ordered boosting, native categoricals |
| Grows trees | level-wise | leaf-wise (deeper, faster) | symmetric |
| Categoricals | needs encoding | native | native, target-statistics based |
| Best at | the default choice, huge ecosystem | very large datasets, speed | many high-cardinality categoricals |

`HistGradientBoosting*` in scikit-learn is essentially LightGBM's histogram-binning
idea reimplemented with the scikit-learn API - which is why we have used it all
course: it composes with `Pipeline`, `ColumnTransformer` and `GroupKFold` without any
adapters.

**What actually matters, in order:**

1. Your validation protocol. A leaky protocol with CatBoost is worse than a clean one
   with a decision tree.
2. Your features.
3. Whether you tuned `learning_rate`/`n_estimators` together.
4. Which library. Distant fourth - we measured p = 0.16.

> Practitioners who switch libraries hoping for a score improvement are almost always
> looking in the wrong place. The gains are in §4's protocol and §7's error analysis.

## §7 - Interpretation

We have a champion. Now: what is it doing, and what may we say about that?

In [ ]:
# Interpreting the champion. Fitted on the §2 split so importance can be measured on
# listings the model never saw.
champion = Pipeline([("prep", make_preprocessor()),
                     ("model", HistGradientBoostingClassifier(random_state=SEED))])
champion.fit(X_tr, y_tr)

# scoring="roc_auc" so the drop is expressed in the metric the client cares about,
# rather than in accuracy on an imbalanced target.
importance = permutation_importance(champion, X_te, y_te, n_repeats=5,
                                    random_state=SEED, scoring="roc_auc")
ranked = pd.Series(importance.importances_mean,
                   index=NUMERIC + CATEGORICAL).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
ranked.head(8).iloc[::-1].plot.barh(ax=ax, color="steelblue")
ax.set(title="permutation importance - AUC lost when a column is shuffled",
       xlabel="Δ AUC")
plt.tight_layout()
plt.show()
# Importance measures this model's reliance on a column. It is not causal, and
# correlated columns split their importance between them, so both look smaller than
# either really is. That caveat belongs in the model card, not in a footnote.
print(ranked.head(6).round(4).to_string())

### `minimum_nights` is back - and this time it is legitimate

The top feature is `minimum_nights` again, at 0.130. In Session 6 that same column
invalidated the price model.

Here it is **exactly the right feature to use.** Catalan regulation treats
short-term tourist accommodation differently from long-stay rental, and a listing
with a 32-night minimum is a *different regulatory object* - often genuinely exempt
from the tourist-licence regime. So minimum stay is not an artifact of our
measurement here; it is a direct signal about which legal category the listing is in.

> **The same column. The same dataset. Invalid for one target and central to
> another.** This is Session 6's lesson stated as concretely as it can be: leakage is
> a relationship between a column, a target and a decision - never a property of a
> column you can flag once and forget.

### What permutation importance does not tell you

It answers: *how much does this model's AUC depend on this column?* It does **not**
say:

- that the column *causes* anything;
- that the column is important **in the world** - only in this fitted model;
- anything reliable when features are correlated. Shuffling `bedrooms` while
  `accommodates` stays intact lets the model recover most of the information, so
  correlated features **share** and therefore understate their importance.

That last point matters here: `accommodates` ranks sixth at 0.004, which is
certainly not because capacity is irrelevant to licensing.

## §8 - Error analysis: who does this model fail?

Aggregate AUC is a summary of 12,626 listings. It tells you nothing about whether the
failures are spread evenly or concentrated on one kind of person.

We are directing inspectors to homes. This section is not optional.

### Predict before you run

We will compute out-of-fold performance **per district**, and check where the top-200
inspection flags land.

**Write your prediction:** which districts will be over-represented among the flags,
and why?

In [ ]:
# TODO: Prediction first, then the analysis.
#
# My prediction: the flags will concentrate in ______________ because ______________
#
# Then:
#   1. Get out-of-fold predicted probabilities for the champion.
#   2. For each district with at least 100 listings, compute AUC and recall.
#   3. Take the top 200 by predicted risk of being unlicensed and compare each
#      district's share of flags with its share of listings.

### Two findings, and the second one is not what anyone predicts

**1. Performance varies by district: AUC 0.861 to 0.961, a gap of 0.10.** The worst
is **Ciutat Vella**, the historic centre - and it is the second-largest district in
the dataset with 2,663 listings. So our headline 0.922 is an average that conceals a
systematically weaker model exactly where short-term rental is most contested.

**2. The flags concentrate in the wealthiest district, not the poorest.**

| district | over-representation |
|---|---:|
| Sarrià-Sant Gervasi | **3.00×** |
| Nou Barris | 2.15× |
| Gràcia | 1.64× |
| Eixample | 0.97× |
| Ciutat Vella | **0.50×** |
| Sant Andreu | 0.37× |

Sarrià-Sant Gervasi is the richest district in Barcelona and receives three times its
proportional share of inspections. Ciutat Vella - the tourist-saturated old city,
lower income, the epicentre of the housing debate - receives **half** its share.

Almost everyone predicts the opposite. The standard fairness narrative for
enforcement models is that they over-police poorer areas, and that is often true. It
is not true here.

> **Fairness is an empirical question about a specific model and dataset, not a
> deduction from a general pattern.** You have to measure it. Had we reasoned instead
> of measuring, we would have written a confident and false paragraph in the report.

### So is the model unfair?

The honest answer is that we cannot tell from this table alone, and the ambiguity is
instructive. At least three readings are consistent with these numbers:

1. **It is working.** Sarrià genuinely has more unlicensed listings per capita, and
   the model found them. Enforcement *should* be concentrated there.
2. **It is exploiting a recording artifact.** Perhaps licences are recorded less
   consistently in Sarrià, so the model has learned where paperwork is missing rather
   than where the law is broken.
3. **It is failing where it matters most.** The model is weakest in Ciutat Vella
   (AUC 0.861), so it may be *under*-flagging there because it cannot tell compliant
   from non-compliant listings well in that district.

Reading 3 is the one that should worry the city, and it is the one the aggregate
number hides entirely.

**What goes in the Model Card:** all three readings, the district table, and a
recommendation that the first month of inspections be sampled partly at random so
that the city can distinguish between them. A model that cannot be audited after
deployment should not be deployed.

## §9 - The regression contrast, and Model Card v1

Before writing the card, one check. Everything above is the classification task.
What do ensembles do on the **price** problem from Session 6?

In [ ]:
# Session 6 already measured Ridge and HistGB on exactly this task, so we do not pay
# for them again - re-running a 170-second fit to reproduce a number we established
# three weeks ago is not thoroughness, it is waste. We fit the one model we have not
# tried on the price problem: a Random Forest.
work = train[train["price_num"].notna()]
work = work[work["price_num"].between(10, 2000)]
# The Session 6 re-scoping, carried forward: short stays only, and the columns that
# encode the booking constraint are out of the feature list.
short = work[work["minimum_nights"] <= 3]
y_price = np.log1p(short["price_num"])
NUM_PRICE = [c for c in NUMERIC if c not in ("minimum_nights", "availability_365")]
price_folds = list(GroupKFold(5).split(short, y_price, groups=short["host_id"]))

price_scores = {}
for name, model in [("Ridge", Ridge()),
                    ("RandomForest (100)", RandomForestRegressor(n_estimators=100,
                                                                 random_state=SEED,
                                                                 n_jobs=-1))]:
    s = cross_val_score(
        Pipeline([("prep", make_preprocessor(NUM_PRICE, CATEGORICAL)), ("model", model)]),
        short[NUM_PRICE + CATEGORICAL], y_price, cv=price_folds, scoring="r2")
    price_scores[name] = s
    print(f"  {name:20s} R2 = {s.mean():.4f} +/- {s.std():.4f}")

# Quoted from Session 6 rather than recomputed, and labelled as such so the source
# of every number on the screen stays traceable.
print(f"  {'HistGB':20s} R2 = 0.5904 +/- 0.0675   (measured in Session 6)")

# The same paired test as the classification side. On the price problem the
# ensembles do not beat Ridge, and saying so plainly is the deliverable.
diff = price_scores["RandomForest (100)"] - price_scores["Ridge"]
test = stats.ttest_rel(price_scores["RandomForest (100)"], price_scores["Ridge"])
signs = "".join("+" if d > 0 else "-" for d in diff)
print(f"\n  RandomForest - Ridge: {diff.mean():+.4f}  signs {signs}  "
      f"p = {test.pvalue:.3f}  "
      f"{'REAL' if test.pvalue < 0.05 else 'within noise'}")
print("  HistGB       - Ridge: -0.0163  signs -+---  p = 0.469  within noise"
      "   (Session 6)")

### Same dataset. Same session. Opposite conclusions.

| task | ensemble vs simple model | verdict |
|---|---|---|
| **classification** (licensed) | HistGB 0.9216 vs LogReg 0.8899 | **real**, p = 0.001 |
| **regression** (price) | RF 0.6050 vs Ridge 0.6068 | noise, p = 0.93 - and *lower* |
| **regression** (price) | HistGB 0.5904 vs Ridge 0.6068 | noise, p = 0.47 - and *lower* |

On the price problem **neither ensemble beats a linear model, and both score lower**.
"Boosting wins on tabular data" is not a law; it is an empirical regularity that
holds on some problems and not others - and you now have the tools to find out which
kind you are looking at.

The plausible reason: after re-scoping we have 6,302 rows, the target is
log-transformed and roughly symmetric, and the relationship between capacity,
location and log-price really is close to linear. There is not much non-linear
structure left for a tree ensemble to find. The licensing target, by contrast, is
driven by *combinations* - room type **and** district **and** minimum stay - which
is exactly what trees represent natively and a linear model cannot.

In [ ]:
# TODO: Write Model Card v1 for the classification champion. Required sections:
#
#   1. Intended use, and the client
#   2. Performance, with uncertainty, and the validation protocol
#   3. Top features, with the caveat about what importance does not prove
#   4. Subgroup performance, including the district table
#   5. Known failure modes
#   6. Conditions under which this model should NOT be used
#   7. What you would need in order to trust it more
#
# This is deliverable M6. It is graded. See 05_Project/MILESTONES.md.

## §10 - Common mistakes

| Mistake | Why it is tempting | What to do instead |
|---|---|---|
| Ranking five means and declaring a winner | the numbers are ordered | RF vs HistGB differed by 0.0001 with p = 0.97 |
| "More trees is always better" | it is true for forests | boosting fell from 0.920 to 0.910 over 800 iterations |
| Tuning `learning_rate` then `n_estimators` | they look like two knobs | they are one coupled decision |
| Switching libraries for a score gain | new library, new hope | XGBoost vs HistGB: p = 0.16 |
| Reporting impurity importance | it is the default `.feature_importances_` | biased toward high-cardinality features; use permutation importance |
| Reading importance as causation | the word "importance" | it measures this model's dependence, nothing more |
| Reporting one aggregate metric | it is the headline number | district AUC ranged 0.861–0.961 |
| Assuming the fairness problem's direction | the usual narrative fits most cases | it over-flagged the *wealthiest* district 3× |
| Removing a feature because it caused trouble before | consistency feels safe | `minimum_nights` was invalid for price and is central here |

## §11 - Reflection

1. Random Forest and HistGB tied on AUC. Name three considerations besides accuracy
   that could decide between two tied models, and say which would matter most to
   Client A.
2. The model is weakest in Ciutat Vella and over-flags Sarrià-Sant Gervasi. Write the
   paragraph you would put in a report to the city, without claiming more than the
   data supports.
3. No ensemble beat Ridge on the price task. What would have to be true about a
   dataset for boosting to help a lot?

## §12 - Knowledge check

1. Thirty bootstrapped trees averaged 0.769 individually and 0.897 combined. What
   property of the individual trees made this work, and when would it not?
2. `RandomForest(30)` beat hand-rolled bagging of 30 trees, 0.908 to 0.897. What
   accounts for the difference?
3. Adding trees improved the forest and degraded the booster. Explain both.
4. Two models score 0.9216 and 0.9216 with p = 0.97. How do you choose?
5. `minimum_nights` was removed from the price model and kept in the licensing model.
   Justify both decisions in one sentence each.

## Summary

- **Bagging works by cancelling variance.** 30 bootstrapped trees, individually
  AUC 0.769 and disagreeing by 0.228 per listing, average to **0.897**. Returns
  diminish as 1/k.
- **Random Forests add decorrelation** on top of averaging - feature subsampling
  makes each tree worse and the ensemble better (0.897 → 0.908).
- **Boosting is gradient descent in function space.** Each tree fits the negative
  gradient of the loss; you have now met gradient descent in Sessions 4, 5 and 7, and
  will again in Session 8.
- **Adding trees means opposite things.** RF 0.906 → 0.922 and plateaus; boosting
  0.920 → 0.910 and degrades. Only one of them needs early stopping.
- `learning_rate` and `n_estimators` are **one coupled decision**.
- **Ensembles genuinely beat single models here** (p = 0.001) - and RF vs HistGB
  (p = 0.97) and HistGB vs XGBoost (p = 0.16) are indistinguishable. That is the
  measured justification for teaching one boosting library.
- **The tie was broken by granularity, not accuracy**: 11,336 distinct scores against
  444, and 1 tied listing against 182. Deployment requirements break statistical ties.
- `minimum_nights` - invalid for the price target - is the **top legitimate feature**
  for licensing. Leakage is never a property of a column.
- **District AUC ranges 0.861 to 0.961.** The model is weakest in Ciutat Vella, the
  district that matters most.
- **The flags concentrate in the wealthiest district (3.0×), not the poorest.**
  Nobody predicted that. Fairness is measured, not deduced.
- **No ensemble beat Ridge on the price task.** "Boosting wins on tabular data" is a
  regularity, not a law.

## Key takeaways

1. When two models tie statistically, the deployment decides.
2. An aggregate metric is an average over people. Look at the subgroups.
3. Measure the fairness question. Do not reason about it.

## Further exploration

**Essential**
- Hastie, Tibshirani & Friedman, *The Elements of Statistical Learning*, ch. 10
  (boosting) and §15 (random forests). https://hastie.su.domains/ElemStatLearn/
- scikit-learn user guide, *Ensembles*:
  https://scikit-learn.org/stable/modules/ensemble.html
- Mitchell et al. (2019), *Model Cards for Model Reporting*, FAT\* - the format used
  in §9. https://arxiv.org/abs/1810.03993

**Recommended**
- Friedman (2001), *Greedy Function Approximation: A Gradient Boosting Machine*,
  Annals of Statistics 29(5) - boosting as functional gradient descent, at source.
- Breiman (2001), *Random Forests*, Machine Learning 45(1) - the decorrelation
  argument from §2.
- Chen & Guestrin (2016), *XGBoost: A Scalable Tree Boosting System*, KDD.

**Advanced**
- Grinsztajn, Oyallon & Varoquaux (2022), *Why do tree-based models still outperform
  deep learning on tabular data?* https://arxiv.org/abs/2207.08815 - read it **before**
  Session 9. It explains why the model you are about to build will probably lose.
- Barocas, Hardt & Narayanan, *Fairness and Machine Learning*, ch. 2–3.
  https://fairmlbook.org/ - the formal vocabulary for §8's three readings.

---

**Next session:** we abandon closed-form solutions and libraries entirely, and build
a neural network from nothing but NumPy - including computing one backpropagation
step by hand.